# Unemployment Analysis in India

In this project, I am analyzing the unemployment situation across different states and regions of India. The dataset contains monthly unemployment data including the unemployment rate, number of employed people, and labour participation rate — broken down by region and area type (Rural/Urban).

My goal is to understand how unemployment changed over time, which months or years had the highest unemployment, and whether there are any visible patterns in the data. I will go through the full data science workflow — loading, exploring, cleaning, handling outliers, and finally visualizing the results.

## Step 1 — Importing the Required Libraries

Before starting anything, I need to import all the libraries I'll be using throughout this project.

- **pandas** — for loading and manipulating the dataset
- **numpy** — for numerical operations
- **matplotlib & seaborn** — for creating static visualizations
- **plotly** — for interactive charts
- **scipy & statsmodels** — for statistical analysis

I also use `%matplotlib inline` so that all plots appear directly inside the notebook without needing to call `plt.show()` separately.

In [72]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats
import statsmodels.api as sm
%matplotlib inline

## Step 2 — Loading the Dataset

Here I am loading the dataset from a CSV file called `Unemployment in India.csv` using pandas `read_csv()`.

This dataset contains monthly unemployment records for different states of India. Once loaded, the data is stored in a DataFrame called `df`, which I'll use for all further analysis.

In [2]:
df=pd.read_csv("Unemployment in India.csv")

## Step 3 — Exploring the Data

Before doing anything with the data, I always like to explore it first to understand what I'm working with. I check:

- **`df.head()`** — shows the first 5 rows so I can see the structure and column names
- **`df.shape`** — tells me how many rows and columns the dataset has (768 rows, 7 columns)
- **`df.info()`** — shows the data types of each column and how many non-null values exist. I noticed that some columns have leading spaces in their names, and the `Date` column is stored as a string (object) instead of datetime — both of which I'll fix in the cleaning step
- **`df.describe()`** — gives me basic statistics like mean, min, max, and standard deviation for the numeric columns, which helps me spot potential outliers early

In [4]:
df.head()

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural


In [5]:
df.shape

(768, 7)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 7 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Region                                    740 non-null    object 
 1    Date                                     740 non-null    object 
 2    Frequency                                740 non-null    object 
 3    Estimated Unemployment Rate (%)          740 non-null    float64
 4    Estimated Employed                       740 non-null    float64
 5    Estimated Labour Participation Rate (%)  740 non-null    float64
 6   Area                                      740 non-null    object 
dtypes: float64(3), object(4)
memory usage: 42.1+ KB


In [7]:
df.describe()

## Step 4 — Cleaning the Data

Raw data is almost never clean, and this dataset is no different. I found a few issues that I need to fix before I can do any proper analysis:

**1. Dropping fully empty rows** — The dataset has 768 rows but only 740 non-null values per column, meaning there are 28 completely empty rows. I remove them using `dropna(how='all')` so they don't interfere with the analysis.

**2. Renaming columns** — The column names have leading spaces (e.g., `' Date'`, `' Estimated Unemployment Rate (%)'`) which makes them hard to work with. I rename them to clean, short names like `Unemployment_Rate`, `Employed`, and `Labour_Participation_Rate`.

**3. Fixing the Date column** — The `Date` column is stored as a plain string. I first strip any extra whitespace, then convert it to a proper datetime format using `pd.to_datetime()` with the format `'%d-%m-%Y'`. This is important because later I need to extract the year and month from it.

**4. Dropping the Frequency column** — This column only contains the value 'Monthly' for every row, so it adds no useful information. I drop it to keep the dataset clean.

**5. Converting Area to category** — The `Area` column only has two values: 'Rural' and 'Urban'. Converting it to a categorical type saves memory and makes it easier to use in groupby operations.

After all these steps, I run `df.info()` again to confirm that everything looks correct — the Date column should now be `datetime64` and there should be no extra columns.

In [3]:
df = df.dropna(how='all')

In [4]:
df.rename(columns={
    ' Estimated Unemployment Rate (%)': 'Unemployment_Rate',
    ' Date': 'Date',
    ' Frequency':'Frequency',
    ' Estimated Employed':'Employed',
    ' Estimated Labour Participation Rate (%)' : 'Labour_Participation_Rate',
    
}, inplace=True)

In [5]:
df['Date'] = df['Date'].str.strip()
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

In [6]:
df.drop('Frequency', axis=1, inplace=True)

In [7]:
df['Area'] = df['Area'].astype('category')

In [9]:
df.info()

## Step 5 — Handling Outliers Using IQR Clipping

From the `describe()` output earlier, I noticed that some numeric columns have extreme values — for example, the max unemployment rate was 76.74% which is unusually high compared to the mean of ~11.8%. These outliers can skew the visualizations and make it hard to see the actual trends.

To handle this, I use the **IQR (Interquartile Range)** method. The idea is simple:
- Calculate Q1 (25th percentile) and Q3 (75th percentile)
- Compute IQR = Q3 - Q1
- Any value below `Q1 - 1.5 * IQR` or above `Q3 + 1.5 * IQR` is considered an outlier
- Instead of deleting those rows, I **clip** the values to the fence boundaries — this keeps all rows but limits the extreme values

I apply this to all three numeric columns: `Unemployment_Rate`, `Employed`, and `Labour_Participation_Rate`.

I print a **BEFORE vs AFTER** comparison to clearly show the effect — after clipping, the max unemployment rate drops from 76.74% to 32.73%, which is much more reasonable for visualization purposes.

In [8]:
numeric_cols = ['Unemployment_Rate', 'Employed', 'Labour_Participation_Rate']

df_cleaned = df.copy()

for col in numeric_cols:

    Q1  = df_cleaned[col].quantile(0.25)
    Q3  = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1


    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    df_cleaned[col] = df_cleaned[col].clip(lower=lower_fence, upper=upper_fence)

print("=== BEFORE ===")
print(df[numeric_cols].describe().round(2))

print("\n=== AFTER ===")
print(df_cleaned[numeric_cols].describe().round(2))

In [69]:
df = df_cleaned

## Step 6 — Final Preview of the Cleaned Dataset

Now that all the cleaning and outlier handling is done, I take one final look at the dataset using `df.head()` to make sure everything looks correct.

At this point the dataset should have:
- Clean column names (no leading spaces)
- `Date` in proper datetime format
- No `Frequency` column
- `Area` as a category
- Outliers clipped in the numeric columns

This cleaned version is what I'll use for all the visualizations going forward.

In [10]:
df.head()

## Step 7 — Visualizing Yearly Trends

Now I start the actual analysis. My first question is: **how did unemployment and employment change year by year?**

I extract the year from the `Date` column using `.dt.year` and plot two bar charts:

1. **Average Unemployment Rate by Year** — This shows whether unemployment went up or down over the years. I expect to see a spike around 2020 due to the COVID-19 pandemic.

2. **Average Employment by Year** — This shows the average number of employed people per year, giving a sense of the overall employment level.

These two charts together give a good high-level picture of how the job market in India changed over time.

In [31]:
sns.barplot(data=df, x=df['Date'].dt.year, y='Unemployment_Rate')

plt.title('Average Unemployment Rate by Year')
plt.xlabel('Year')
plt.ylabel('Unemployment Rate (%)')

plt.show()

In [32]:
sns.barplot(data=df, x=df['Date'].dt.year, y='Employed')

plt.title('Average Employment by Year')
plt.xlabel('Year')
plt.ylabel('Number of Employed People')

plt.show()

## Step 8 — Visualizing Monthly Trends

After looking at yearly trends, I want to go deeper and check: **does unemployment follow any seasonal pattern across months?**

To do this, I extract the month name from the `Date` column using `.dt.month_name()`. I also define a custom `month_order` list so that the months appear in the correct calendar order (January to December) on the chart — otherwise pandas would sort them alphabetically.

I then plot two bar charts:

1. **Unemployment Rate by Month** — This helps me see if certain months consistently have higher or lower unemployment. For example, agricultural seasons or festival periods might affect employment patterns.

2. **Employed People by Month (in Lakhs)** — I divide the `Employed` column by 100,000 to convert the numbers to Lakhs, which makes the y-axis much easier to read. This chart shows which months tend to have more people employed on average.

In [70]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

df['Month'] = pd.Categorical(df['Date'].dt.month_name(), categories=month_order, ordered=True)

In [71]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df,
    x='Month',
    y='Unemployment_Rate'
)

plt.ylabel('Unemployment Rate (%)')
plt.xticks(rotation=45)
plt.show()

In [81]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df,
    x='Month',
    y=df['Employed'] / 100000
)

plt.ylabel('Employed People (Lakhs)')
plt.show()